In [1]:
# =============================================================================
# Multimodal Demographic Analysis System
# - Voice Age Detection (CommonVoice, CSV with valid "age")
# - Fingerprint Gender Detection (SOCOFing, parse "M"/"F" from filename)
# =============================================================================

import os
import sys
import random
import numpy as np
import pandas as pd
import librosa
import cv2
import pickle
import tensorflow as tf
import ipywidgets as widgets
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, clear_output, HTML
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------- PATHS & CONFIG -------------------------
# Voice dataset CSV (with columns: filename, age), no missing or "N/A" rows
VOICE_EVAL_CSV = "/content/drive/MyDrive/multimodal_project/dataset/CommonVoice/cv-other-test.csv"
# Base directory for voice files if CSV paths are relative
VOICE_BASE_DIR = "/content/drive/MyDrive/multimodal_project/dataset/CommonVoice/cv-other-test"

# Fingerprint "Real" directory in SOCOFing (with files named like "100__M_...")
FINGERPRINT_BASE_DIR = "/content/drive/MyDrive/multimodal_project/dataset/SOCOFing/SOCOFing/Altered/Altered-Easy"

# Pre-trained models & label map
VOICE_MODEL_PATH = "/content/drive/MyDrive/MultimodalDemographicAnalysis/voice_model.h5"
GENDER_MODEL_PATH = "/content/drive/MyDrive/MultimodalDemographicAnalysis/gender_model.h5"
VOICE_LABEL_MAP_PATH = "/content/drive/MyDrive/MultimodalDemographicAnalysis/voice_label_map.pkl"

# Max subset for evaluation
MAX_EVAL_SAMPLES = 10

# ------------------------- 0. CHECK & LOAD MODELS -------------------------
if not (os.path.exists(VOICE_MODEL_PATH) and os.path.exists(GENDER_MODEL_PATH) and os.path.exists(VOICE_LABEL_MAP_PATH)):
    sys.exit("Error: Pre-trained models or label map not found in Drive. Please place them correctly.")

try:
    voice_model = tf.keras.models.load_model(VOICE_MODEL_PATH)
    gender_model = tf.keras.models.load_model(GENDER_MODEL_PATH)
    with open(VOICE_LABEL_MAP_PATH, "rb") as f:
        voice_label_map = pickle.load(f)
    print("Models & label map loaded successfully.")
except Exception as e:
    sys.exit(f"Error loading models or label map: {e}")

# ------------------------- 1. VOICE DATA: LOAD CSV & FILTER -------------------------
if not os.path.exists(VOICE_EVAL_CSV):
    sys.exit(f"Voice CSV not found at {VOICE_EVAL_CSV}")

voice_df = pd.read_csv(VOICE_EVAL_CSV)
# Drop any missing/invalid "age"
voice_df = voice_df.dropna(subset=["age"])
voice_df = voice_df[voice_df["age"].astype(str).str.lower() != "n/a"]
print(f"Valid voice rows (with 'age'): {len(voice_df)}")

def resolve_voice_path(path):
    if os.path.isabs(path):
        return path
    return os.path.join(VOICE_BASE_DIR, path)

# For the UI, pick up to 3 random samples
sample_voice_df = voice_df.sample(n=min(3, len(voice_df)), random_state=42)

# ------------------------- 2. FINGERPRINT DATA: SCAN "REAL" DIR, PARSE M/F -------------------------
def parse_gender_from_filename(fname):
    # e.g. "100__M_Left_index_finger.BMP" => 'M'
    parts = fname.split("__")
    if len(parts) < 2:
        return None
    letter = parts[1][0].upper()
    if letter in ["M", "F"]:
        return letter
    return None

finger_list = []
for root, dirs, files in os.walk(FINGERPRINT_BASE_DIR):
    for file in files:
        if file.lower().endswith(".bmp"):
            gender_letter = parse_gender_from_filename(file)
            if gender_letter in ["M", "F"]:
                full_path = os.path.join(root, file)
                finger_list.append((full_path, gender_letter))

print(f"Found {len(finger_list)} valid fingerprint files with M/F in filename in Real folder.")

# For the UI, pick up to 3 random samples
random.seed(42)
finger_samples = random.sample(finger_list, min(3, len(finger_list)))

# ------------------------- 3. INFERENCE FUNCTIONS -------------------------
def process_audio(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000, duration=4)
        y = librosa.effects.trim(y, top_db=20)[0]
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=64)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64))
        features = np.vstack([mfcc, mel]).T
        MAX_SEQ = 160
        feat_dim = features.shape[1]
        X = np.zeros((MAX_SEQ, feat_dim))
        seq_len = min(features.shape[0], MAX_SEQ)
        X[:seq_len, :] = features[:seq_len, :]
        X = np.expand_dims(X, axis=0)
        return X
    except Exception as e:
        print("Audio processing error:", e)
        return None

def predict_age(file_path):
    abs_path = resolve_voice_path(file_path)
    X = process_audio(abs_path)
    if X is None:
        return "Error reading audio."
    proba = voice_model.predict(X, verbose=0)[0]
    inv_label_map = {v: k for k, v in voice_label_map.items()}
    sorted_idx = np.argsort(proba)[::-1]
    lines = []
    for idx in sorted_idx:
        label = inv_label_map.get(idx, f"Class_{idx}")
        lines.append(f"{label}: {proba[idx]*100:.2f}%")
    return "\n".join(lines)

def load_image(img_path):
    try:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (96,96))
        img = img.astype("float32") / 255.0
        img = np.expand_dims(img, axis=-1)
        img = np.expand_dims(img, axis=0)
        return img
    except Exception as e:
        print("Image processing error:", e)
        return None

def predict_gender(img_path):
    X = load_image(img_path)
    if X is None:
        return "Error reading fingerprint."
    proba = gender_model.predict(X, verbose=0)[0]
    idx = np.argmax(proba)
    return "Male" if idx == 0 else "Female"

# ------------------------- 4. EVALUATION (Subset of 10) -------------------------
def evaluate_voice_model(df, max_samples=10):
    df = df.head(max_samples)
    y_true, y_pred = [], []
    for _, row in df.iterrows():
        path = row['filename']
        actual = row['age']
        pred_str = predict_age(path)
        top_pred = pred_str.split(":")[0] if "Error reading audio." not in pred_str else "N/A"
        y_true.append(actual)
        y_pred.append(top_pred)
    if len(y_true) == 0:
        return None, None
    rep = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)
    return rep, cm

def evaluate_fingerprint_model(f_list, max_samples=10):
    subset = f_list[:max_samples]
    y_true, y_pred = [], []
    for (path, letter) in subset:
        # Actual label from letter
        actual_label = "Male" if letter.upper() == "M" else "Female"
        pred = predict_gender(path)
        y_true.append(actual_label)
        y_pred.append(pred)
    if len(y_true) == 0:
        return None, None
    rep = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)
    return rep, cm

def plot_confusion_matrix(cm, title):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(title)
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

voice_report, voice_cm = evaluate_voice_model(voice_df, max_samples=MAX_EVAL_SAMPLES)
finger_report, finger_cm = evaluate_fingerprint_model(finger_list, max_samples=MAX_EVAL_SAMPLES)

# ------------------------- 5. BUILD UI (IPYWIDGETS) -------------------------
display(HTML("""
<style>
    .widget-label { font-size: 16px; color: #2a9d8f; }
    .widget-button { background-color: #264653 !important; color: #ffffff !important; }
    .widget-tab { background-color: #e9c46a; }
    .my-header { font-size: 24px; color: #e76f51; text-align: center; }
    .sample-box { border: 1px solid #ccc; padding: 5px; margin: 5px; }
</style>
"""))

tab = widgets.Tab()

# --- Voice Age Detection Tab ---
voice_upload = widgets.FileUpload(accept=".wav,.mp3", multiple=False)
voice_button = widgets.Button(description="Predict Age", button_style='success')
voice_out = widgets.Output()

def on_voice_predict(b):
    with voice_out:
        clear_output()
        if voice_upload.value:
            for fname, file_info in voice_upload.value.items():
                temp_path = f"/tmp/{fname}"
                with open(temp_path, "wb") as f:
                    f.write(file_info["content"])
                pred = predict_age(temp_path)
                print(f"Uploaded File: {fname}")
                print("Prediction:\n" + pred)
                display(widgets.Audio(value=file_info["content"], format='mp3', autoplay=False))
        else:
            print("No file uploaded.")

voice_button.on_click(on_voice_predict)

voice_samples_box = widgets.VBox([widgets.HTML("<h4>Sample Voice Test Cases (3, valid 'age')</h4>")])
for _, row in sample_voice_df.iterrows():
    file_path = row['filename']
    actual_age = row['age']
    pred_str = predict_age(file_path)
    top_pred = pred_str.split(":")[0] if "Error reading audio." not in pred_str else "N/A"
    sample_html = widgets.HTML(
        f"<div class='sample-box'>"
        f"<b>File:</b> {file_path}<br>"
        f"<b>Actual:</b> {actual_age} &nbsp;&nbsp; <b>Predicted:</b> {top_pred}<br>"
        f"<pre>{pred_str}</pre>"
        f"</div>"
    )
    abs_path = file_path if os.path.isabs(file_path) else os.path.join(VOICE_BASE_DIR, file_path)
    audio_bytes = None
    if os.path.exists(abs_path):
        with open(abs_path, "rb") as f:
            audio_bytes = f.read()
    if audio_bytes:
        audio_widget = widgets.Audio(value=audio_bytes, format='mp3', autoplay=False)
        voice_samples_box.children += (widgets.VBox([sample_html, audio_widget]),)
    else:
        voice_samples_box.children += (sample_html,)

voice_tab = widgets.VBox([
    widgets.HTML("<h2 style='color:#2a9d8f;'>Voice Age Detection</h2>"),
    voice_upload,
    voice_button,
    voice_out,
    widgets.HTML("<hr>"),
    voice_samples_box
])

# --- Fingerprint Gender Detection Tab ---
finger_upload = widgets.FileUpload(accept=".bmp,.png,.jpg", multiple=False)
finger_button = widgets.Button(description="Predict Gender", button_style='info')
finger_out = widgets.Output()

def on_finger_predict(b):
    with finger_out:
        clear_output()
        if finger_upload.value:
            for fname, file_info in finger_upload.value.items():
                temp_path = f"/tmp/{fname}"
                with open(temp_path, "wb") as f:
                    f.write(file_info["content"])
                pred = predict_gender(temp_path)
                print(f"Uploaded File: {fname}")
                print("Prediction:", pred)
                display(widgets.Image(value=file_info["content"], format='png'))
        else:
            print("No file uploaded.")

finger_button.on_click(on_finger_predict)

finger_samples_box = widgets.VBox([widgets.HTML("<h4>Sample Fingerprint Test Cases (3, with M/F in filename)</h4>")])
for fp_path, letter in finger_samples:
    actual_label = "Male" if letter.upper() == "M" else "Female"
    pred_label = predict_gender(fp_path)
    sample_html = widgets.HTML(
        f"<div class='sample-box'>"
        f"<b>File:</b> {fp_path}<br>"
        f"<b>Actual:</b> {actual_label} &nbsp;&nbsp; <b>Predicted:</b> {pred_label}"
        f"</div>"
    )
    if os.path.exists(fp_path):
        with open(fp_path, "rb") as f:
            img_data = f.read()
        img_widget = widgets.Image(value=img_data, format='bmp', width=120, height=120)
        finger_samples_box.children += (widgets.VBox([sample_html, img_widget]),)
    else:
        finger_samples_box.children += (sample_html,)

finger_tab = widgets.VBox([
    widgets.HTML("<h2 style='color:#264653;'>Fingerprint Gender Detection</h2>"),
    finger_upload,
    finger_button,
    finger_out,
    widgets.HTML("<hr>"),
    finger_samples_box
])

# --- Evaluation Tab ---
eval_out = widgets.Output()

def on_eval_click(b):
    with eval_out:
        clear_output()
        print(f"Overall Evaluation for Voice (up to {MAX_EVAL_SAMPLES} samples):")
        if voice_report is not None and voice_cm is not None:
            display(pd.DataFrame(voice_report).T)
            print("\nConfusion Matrix (Voice):")
            plot_confusion_matrix(voice_cm, "Voice Age Detection")
        else:
            print("No voice evaluation data.")

        print(f"\nOverall Evaluation for Fingerprint (up to {MAX_EVAL_SAMPLES} samples):")
        if finger_report is not None and finger_cm is not None:
            display(pd.DataFrame(finger_report).T)
            print("\nConfusion Matrix (Fingerprint):")
            plot_confusion_matrix(finger_cm, "Fingerprint Gender Detection")
        else:
            print("No fingerprint evaluation data.")

eval_button = widgets.Button(description="Show Overall Evaluation Metrics", button_style='warning')
eval_button.on_click(on_eval_click)

eval_tab = widgets.VBox([
    widgets.HTML("<h2 style='color:#e76f51;'>Overall Evaluation Metrics</h2>"),
    eval_button,
    eval_out
])

# --- Info Tab ---
info_tab = widgets.HTML(f"""
<h2 style='color:#f4a261;'>System Information</h2>
<p>This system uses:
<br>- <b>Voice CSV:</b> {VOICE_EVAL_CSV} (valid 'age' only, no N/A).
<br>- <b>Fingerprint 'Real' Folder:</b> {FINGERPRINT_BASE_DIR}, parsing 'M'/'F' from filenames.
<br>We pick 3 samples for the UI demonstration, and up to {MAX_EVAL_SAMPLES} for overall evaluation.
<br>Models are loaded from:
<ul>
<li>Voice Model: {VOICE_MODEL_PATH}</li>
<li>Fingerprint Model: {GENDER_MODEL_PATH}</li>
<li>Voice Label Map: {VOICE_LABEL_MAP_PATH}</li>
</ul>
</p>
""")

# Combine Tabs
tabs = widgets.Tab()
tabs.children = [voice_tab, finger_tab, eval_tab, info_tab]
tabs.set_title(0, "Voice Age Detection")
tabs.set_title(1, "Fingerprint Gender Detection")
tabs.set_title(2, "Evaluation Metrics")
tabs.set_title(3, "Info")

display(tabs)


SystemExit: Error: Pre-trained models or label map not found in Drive. Please place them correctly.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
